In [0]:
dbutils.widgets.text("catalog", "dev")
dbutils.widgets.text("schema", "bikeshare")
dbutils.widgets.text("volume", "raw")
dbutils.widgets.text("feeds", "station_status")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
feeds_wanted = [f.strip() for f in dbutils.widgets.get("feeds").split(",") if f.strip()]

VOL = f"/Volumes/{catalog}/{schema}/{volume}"
print(VOL, feeds_wanted)

In [0]:
import time
import requests

DISCOVERY_URL = "https://sharedmobility.ch/gbfs.json"   # GBFS 2.0, no key needed

LOG_HEADERS = ("content-length", "content-encoding", "transfer-encoding", "age", "via",
               "x-cache", "server", "date", "last-modified")

def get_json(url, attempts=3, wait_s=5, headers=None):
    for i in range(attempts):
        r = None
        try:
            r = requests.get(url, headers=headers, timeout=(10, 60), stream=True)
            r.raise_for_status()
            return r.json()          # the body is read here, so a cut-off download raises here
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Attempt {i + 1}/{attempts} failed for {url}: {e}")
            if r is not None:
                print({k: v for k, v in r.headers.items() if k.lower() in LOG_HEADERS})
            if i == attempts - 1:
                raise
            time.sleep(wait_s * (i + 1))


# Fallback source for station_status: GBFS 2.3 per-system feeds (16 Nov 2026: the combined v1 file
# was cut off mid-body by the server while the per-system files were complete).
V2_URL = "https://sharedmobility.ch/v2/gbfs"
V2_HEADERS = {"Authorization": "gbfs@sharedmobility.ch"}   # v2 wants a contact email here (SFOE docs)

def bike_systems():
    """Docked bike systems in scope: single definition is the pipeline table ref_bike_systems."""
    return [r.provider_id for r in spark.table(f"{catalog}.{schema}.ref_bike_systems").collect()]

def pull_status_v2(systems):
    """station_status from the v2 per-system feeds, reshaped like the v1 file (same 8 station
    fields, v1 station_id = '<system>:<id>'), so bronze and silver need no change."""
    top = get_json(V2_URL, attempts=2, headers=V2_HEADERS)
    urls = {s["id"]: s["url"] for s in top["systems"]}
    stations, ttls, failed = [], [], []
    for sid in systems:
        try:
            disc = get_json(urls[sid], attempts=2, headers=V2_HEADERS)
            lang = next(iter(disc["data"]))
            feed_urls = {f["name"]: f["url"] for f in disc["data"][lang]["feeds"]}
            doc = get_json(feed_urls["station_status"], attempts=2, headers=V2_HEADERS)
        except Exception as e:          # one failing system must not lose the others
            failed.append(sid)
            print(f"v2 system {sid} failed: {type(e).__name__}: {e}")
            continue
        if doc.get("ttl") is not None:
            ttls.append(doc["ttl"])
        for s in doc["data"]["stations"]:
            stations.append({
                "is_installed": s.get("is_installed"),
                "is_renting": s.get("is_renting"),
                "is_returning": s.get("is_returning"),
                "last_reported": s.get("last_reported"),
                "num_bikes_available": s.get("num_bikes_available"),
                "num_docks_available": s.get("num_docks_available"),
                "provider_id": sid,
                "station_id": f"{sid}:{s['station_id']}",
            })
    if not stations:
        raise RuntimeError(f"v2 fallback returned no stations (failed systems: {failed})")
    if failed:
        print(f"WARNING: v2 snapshot is partial, missing systems: {failed}")
    # only the three keys bronze reads: an extra top-level key would be a schema change for Auto Loader
    return {"last_updated": int(time.time()),
            "ttl": min(ttls) if ttls else None,
            "data": {"stations": stations}}

disc = get_json(DISCOVERY_URL)
lang = next(iter(disc["data"]))
feeds = {f["name"]: f["url"] for f in disc["data"][lang]["feeds"]}

unknown = [f for f in feeds_wanted if f not in feeds]
if unknown:
    raise ValueError(f"Unknown feeds: {unknown}. Available: {sorted(feeds)}")
feeds

In [0]:
import json
from datetime import datetime, timezone

ts = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%SZ")
print(f"Timestamp: {ts}")

for name in feeds_wanted:
    suffix = ""
    try:
        payload = get_json(feeds[name])
    except Exception as e:
        if name != "station_status":
            raise
        print(f"v1 {name} failed ({type(e).__name__}); falling back to GBFS v2 per-system feeds")
        payload = pull_status_v2(bike_systems())
        suffix = "_v2"                 # marks the file in the file name (bronze keeps file_name)
    folder = f"{VOL}/{name}"
    dbutils.fs.mkdirs(folder)
    path = f"{folder}/{name}_{ts}{suffix}.json"
    with open(path, "w") as fh:
        json.dump(payload, fh)
    n_items = {k: len(v) for k, v in payload["data"].items() if isinstance(v, list)}
    print(name, n_items, "->", path)
